# Notebook for testing out various models

### Models to Test : 
* ARIMA
* SARIMAX
* Prophet (formerly FBProphet)
* Random Forest & XGBoost
* LSTM
* GRU
* Transformer Models - BERT

In [9]:
# !pip install xgboost
!pip install prophet
# !pip install torch
# !pip install lightgbm

Defaulting to user installation because normal site-packages is not writeable
                                              0.0/13.3 MB ? eta -:--:--
     -                                        0.3/13.3 MB 7.2 MB/s eta 0:00:02
     --                                       0.9/13.3 MB 9.4 MB/s eta 0:00:02
     ---                                      1.3/13.3 MB 10.2 MB/s eta 0:00:02
     -----                                    1.9/13.3 MB 10.1 MB/s eta 0:00:02
     -------                                  2.4/13.3 MB 10.9 MB/s eta 0:00:01
     -------                                  2.5/13.3 MB 10.6 MB/s eta 0:00:02
     -------                                  2.5/13.3 MB 10.6 MB/s eta 0:00:02
     -------                                  2.5/13.3 MB 10.6 MB/s eta 0:00:02
     -------                                  2.5/13.3 MB 10.6 MB/s eta 0:00:02
     -------                                  2.6/13.3 MB 5.7 MB/s eta 0:00:02
     ---------                                3.0/13.


[notice] A new release of pip is available: 23.1.2 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip


## Libraries

In [38]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from prophet import Prophet
import torch

## Data

In [39]:
data = pd.read_csv("../data/MSFT.csv")
data.describe()

,Open,High,Low,Close,Volume,Dividends,Stock Splits,S_3,S_9,S_18,...,RSI,Bollinger_Mid,Bollinger_Upper,Bollinger_Lower,Volume_Rolling_Mean,Volume_Change,Overall_Mean,Overall_Min,Overall_Max,Target
count,4624.000000,4624.000000,4624.000000,4624.000000,4.624000e+03,4624.000000,4624.0,4624.000000,4624.000000,4624.000000,...,4624.000000,4624.000000,4624.000000,4624.000000,4.624000e+03,4624.000000,4.624000e+03,4624.000000,4.624000e+03,4624.000000
mean,97.668329,98.619755,96.685852,97.700532,4.366770e+07,0.005433,0.0,97.609331,97.342447,96.949257,...,54.807424,96.861425,101.770685,91.952166,4.370419e+07,0.057563,9.745277e+01,11.263443,4.425700e+02,97.791951
std,108.590577,109.614455,107.505575,108.623913,2.823090e+07,0.049509,0.0,108.500638,108.165280,107.683656,...,15.826383,107.574763,113.063350,102.185168,2.351244e+07,0.404716,1.421239e-14,0.000000,5.684957e-14,108.736203
min,11.300616,11.612868,11.055272,11.263443,7.425600e+06,0.000000,0.0,11.325397,11.774778,12.167160,...,6.806137,12.228455,13.246610,10.709566,1.346000e+07,-0.808979,9.745277e+01,11.263443,4.425700e+02,11.263443
25%,21.461361,21.622067,21.249251,21.470865,2.491638e+07,0.000000,0.0,21.437580,21.443548,21.446987,...,43.528802,21.458545,22.347417,20.431159,2.631419e+07,-0.178469,9.745277e+01,11.263443,4.425700e+02,21.472810
50%,39.905386,40.232621,39.613704,39.945398,3.599870e+07,0.000000,0.0,39.904758,39.968141,39.665543,...,54.958112,39.629938,41.709248,37.576866,3.733673e+07,-0.013001,9.745277e+01,11.263443,4.425700e+02,39.958326
75%,137.909117,138.920995,136.130203,137.602917,5.406152e+07,0.000000,0.0,137.679460,137.515888,135.500311,...,66.380252,135.332247,140.676406,128.992779,5.569644e+07,0.204908,9.745277e+01,11.263443,4.425700e+02,137.877995
max,440.850006,443.399994,439.369995,441.579987,5.910522e+08,0.750000,0.0,438.439992,427.239997,426.472778,...,99.111648,425.885500,441.545030,415.076599,2.678342e+08,6.234345,9.745277e+01,11.263443,4.425700e+02,442.570007


In [40]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4624 entries, 0 to 4623
Data columns (total 31 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Date                 4624 non-null   object 
 1   Open                 4624 non-null   float64
 2   High                 4624 non-null   float64
 3   Low                  4624 non-null   float64
 4   Close                4624 non-null   float64
 5   Volume               4624 non-null   int64  
 6   Dividends            4624 non-null   float64
 7   Stock Splits         4624 non-null   float64
 8   S_3                  4624 non-null   float64
 9   S_9                  4624 non-null   float64
 10  S_18                 4624 non-null   float64
 11  lag_1                4624 non-null   float64
 12  lag_2                4624 non-null   float64
 13  lag_3                4624 non-null   float64
 14  Rolling_Mean         4624 non-null   float64
 15  Rolling_Min          4624 non-null   f

In [41]:
data

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,S_3,S_9,...,RSI,Bollinger_Mid,Bollinger_Upper,Bollinger_Lower,Volume_Rolling_Mean,Volume_Change,Overall_Mean,Overall_Min,Overall_Max,Target
0,2006-01-31 00:00:00-05:00,19.679367,20.010764,19.651163,19.848591,94841300,0.0,0.0,19.728721,19.029882,...,65.753582,19.034545,19.762921,18.306170,92388520.0,-0.088057,97.452772,11.263443,442.570007,19.771021
1,2006-02-01 00:00:00-05:00,19.714612,19.792173,19.573592,19.771021,68448800,0.0,0.0,19.787479,19.109794,...,60.806861,19.076851,19.873503,18.280200,94263860.0,-0.278281,97.452772,11.263443,442.570007,19.517181
2,2006-02-02 00:00:00-05:00,19.721660,19.735762,19.425518,19.517181,55073400,0.0,0.0,19.712264,19.209292,...,57.336770,19.101882,19.921680,18.282084,91376680.0,-0.195407,97.452772,11.263443,442.570007,19.418474
3,2006-02-03 00:00:00-05:00,19.376167,19.531290,19.277454,19.418474,75022700,0.0,0.0,19.568892,19.302522,...,54.641929,19.121272,19.952247,18.290297,79477080.0,0.362231,97.452772,11.263443,442.570007,19.157579
4,2006-02-06 00:00:00-05:00,19.397314,19.418467,19.101171,19.157579,60170500,0.0,0.0,19.364412,19.372248,...,52.284134,19.130438,19.958625,18.302251,70711340.0,-0.197969,97.452772,11.263443,442.570007,18.995409
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4619,2024-06-07 00:00:00-04:00,426.200012,426.279999,423.000000,423.850006,13621700,0.0,0.0,424.126668,421.251116,...,53.910632,422.018918,434.639662,409.398174,15460920.0,-0.083411,97.452772,11.263443,442.570007,427.869995
4620,2024-06-10 00:00:00-04:00,424.700012,428.079987,423.890015,427.869995,14003000,0.0,0.0,425.413330,420.978892,...,52.784505,422.712755,434.997026,410.428483,14764580.0,0.027992,97.452772,11.263443,442.570007,432.679993
4621,2024-06-11 00:00:00-04:00,425.480011,432.820007,425.250000,432.679993,14551100,0.0,0.0,428.133331,421.368890,...,53.910602,423.698000,435.854008,411.541992,14805020.0,0.039142,97.452772,11.263443,442.570007,441.059998
4622,2024-06-12 00:00:00-04:00,435.320007,443.399994,433.250000,441.059998,22366200,0.0,0.0,433.869995,424.301110,...,59.861540,424.960500,438.795929,411.125070,15880660.0,0.537080,97.452772,11.263443,442.570007,441.579987


## Data Preparation

In [42]:
X = data.drop('Close',axis = 1)
y = data['Close']

In [43]:
X_train,X_test,y_train,y_test =  train_test_split(X,y,test_size = 0.3,random_state = 101)

### ARIMA

In [58]:
arima_data = data[['Date','Close']].set_index("Date")
print(arima_data)

                                Close
Date                                 
2006-01-31 00:00:00-05:00   19.848591
2006-02-01 00:00:00-05:00   19.771021
2006-02-02 00:00:00-05:00   19.517181
2006-02-03 00:00:00-05:00   19.418474
2006-02-06 00:00:00-05:00   19.157579
...                               ...
2024-06-07 00:00:00-04:00  423.850006
2024-06-10 00:00:00-04:00  427.869995
2024-06-11 00:00:00-04:00  432.679993
2024-06-12 00:00:00-04:00  441.059998
2024-06-13 00:00:00-04:00  441.579987

[4624 rows x 1 columns]


In [59]:
model = ARIMA(arima_data, order = (1,1,1))
model_fit = model.fit()
forecast = model_fit.forecast(steps = 30)
print(forecast)

C:\Users\Rizen3\AppData\Roaming\Python\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Rizen3\AppData\Roaming\Python\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Rizen3\AppData\Roaming\Python\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


4624    441.082892
4625    440.927113
4626    440.878295
4627    440.862997
4628    440.858202
4629    440.856700
4630    440.856229
4631    440.856082
4632    440.856035
4633    440.856021
4634    440.856016
4635    440.856015
4636    440.856014
4637    440.856014
4638    440.856014
4639    440.856014
4640    440.856014
4641    440.856014
4642    440.856014
4643    440.856014
4644    440.856014
4645    440.856014
4646    440.856014
4647    440.856014
4648    440.856014
4649    440.856014
4650    440.856014
4651    440.856014
4652    440.856014
4653    440.856014
Name: predicted_mean, dtype: float64


C:\Users\Rizen3\AppData\Roaming\Python\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\Rizen3\AppData\Roaming\Python\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [ ]:
plt.plot(series, label='Original Data')
plt.plot(np.arange(len(series), len(series) + len(forecast)), forecast, label='Forecasted Values')
plt.xlabel('Time')
plt.ylabel('Value')
plt.title('ARIMA Forecast')
plt.legend()
plt.show()